# ДЗ 6. Очистка orders.csv в Pandas
Срок: **до воскресенья 22.11, 24:00**.
Сдача: папка `hw06` репозитория `dsa-homework` (`hw06.ipynb`): `git add hw06` → `git commit` → `git push`, ссылку прикрепите в LMS.

Каждый ответ: **число и вывод для менеджера** (в print или Markdown-ячейке). Менеджер не читает код: вывод отвечает, что видно в данных и что делать.

Инструменты: всё с уроков 1–6. `groupby`, `pivot_table`, `merge` пока не используем (урок 7). Небольшой цикл по городам или месяцам допустим. Если берёте непройденный инструмент, объясните его в комментарии.

- **[База]** обязательно для всех · **[Стандарт]** сколько получится · **[Челлендж]** для тех, кому легко.

## [Для всех] Чтение кода: без запуска
Сначала напишите ответ и «почему», потом запустите.
```python
import pandas as pd
# 1
s = pd.Series([1, None, 3])
print(s.isna().sum(), s.fillna(0).mean())
# 2
print(pd.Series([" a", "A ", "b"]).str.strip().str.lower().nunique())
# 3
df = pd.DataFrame({"k": [1, 1, 2]})
df.drop_duplicates()
print(len(df))
# 4
print(pd.to_numeric(pd.Series(["1 000", "5"]).str.replace(" ", "")).sum())
# 5
d = pd.to_datetime(pd.Series(["2026-09-30 21:00", "2026-10-01 08:00"]))
print(d.dt.month.tolist())
```

Ваши ответы и объяснения:

1. pd.Series создает одномерный массив из элементов 1, None, 3 далее метод isna() проверяет массив на пустые значения и находит один пропуск None поэтому sum() вернет 1, затем fillna(0) заменяет пропуск на ноль и массив становится 1.0, 0.0, 3.0 после чего mean() считает среднее арифметическое всех трех чисел и выходит 1.3333333333333333 итоговым ответом выйдет: 1 1.3333333333333333
2. pd.Series создает одномерный массив из строк с лишними пробелами и разными регистрами затем метод str.strip() убирает пробелы по краям а str.lower() переводит все буквы в нижний регистр из за чего элементы становятся одинаковыми "a", "a", "b", далее метод nunique() считает количество уникальных значений в массиве и выводит 2 итоговый ответ: 2
3. pd.DataFrame создает таблицу с колонкой k и тремя элементами 1, 1, 2 далее вызывается метод drop_duplicates() который удаляет дубликаты но он не перезаписывает саму переменную df а просто возвращает копию в никуда так как нет присваивания df =, из за этого исходный датафрейм не меняется и len(df) выведет 3 итоговый ответ: 3
4. pd.Series создает массив из двух строк "1 000" и "5" далее метод str.replace(" ", "") убирает пробел внутри первой строки и она превращается в "1000" затем функция pd.to_numeric() переводит эти строки в обычные числа 1000 и 5 а метод sum() их складывает окончательный ответ: 1005
5. функция pd.to_datetime() переводит две строки с датами в специальный формат даты и времени далее через атрибут dt.month мы вытаскиваем номер месяца из каждой даты для сентября это 9 а для октября 10, затем метод tolist() переводит полученный массив в обычный пайтоновский список итоговый результат: [9, 10]

In [10]:
# Загрузка orders.csv: в Colab нажмите «Выбрать файлы» и укажите orders.csv из LMS.
# Если файл уже лежит рядом с ноутбуком, загрузка пропускается.
import os
if not os.path.exists("orders.csv"):
    try:
        from google.colab import files
        files.upload()
    except ImportError:
        print("Положите orders.csv в одну папку с ноутбуком")
print("orders.csv на месте:", os.path.exists("orders.csv"))

orders.csv на месте: True


## [База] 1. Функция очистки
Напишите функцию `clean(path)`, которая читает orders.csv и делает все шаги урока: суммы в числа, города через словарь, дубли, даты, без TEST. Отменённые заказы **не** удаляйте: они нужны для доли отмен.
После каждого шага функция печатает, сколько строк осталось. Проверки внутри функции: у `amount` нет NaN, у `city` нет NaN, `order_id` уникален.
Сколько строк в результате?

In [11]:
# 1import pandas as pd

import pandas as pd 
def clean(path):
    df = pd.read_csv(path)
    print("Исходно строк:", len(df))

    df["amount"] = pd.to_numeric(df["amount"].str.replace(" ", ""))
    print("После перевода amount в числа:", len(df))

    CITY = {
        "алматы": "Алматы",
        "almaty": "Алматы",
        "астана": "Астана",
        "astana": "Астана",
        "шымкент": "Шымкент",
        "shymkent": "Шымкент",
        "караганда": "Караганда",
        "karaganda": "Караганда",
        "актобе": "Актобе",
        "aktobe": "Актобе",
        "атырау": "Атырау",
        "atyrau": "Атырау"
    }
    df["city"] = df["city"].str.strip().str.lower().map(CITY)
    print("После очистки городов:", len(df))

    df = df.drop_duplicates()
    print("После удаления дублей:", len(df))

    df["order_date"] = pd.to_datetime(df["order_date"])
    print("После перевода дат:", len(df))

    df = df[df["customer_id"] != "TEST"]
    print("После удаления TEST:", len(df))

    print("Проверка NaN в amount:", df["amount"].isna().sum())
    print("Проверка NaN в city:", df["city"].isna().sum())
    print("Проверка уникальности order_id:", df["order_id"].is_unique)

    return df

df = clean("orders.csv")
print("Итого строк:", len(df))

Исходно строк: 5463
После перевода amount в числа: 5463
После очистки городов: 5463
После удаления дублей: 5283
После перевода дат: 5283
После удаления TEST: 5280
Проверка NaN в amount: 0
Проверка NaN в city: 0
Проверка уникальности order_id: True
Итого строк: 5280


таблица очищена от текстовых сумм с пробелами, 14 вариантов написания городов сведены к 6 городам через словарь, удалены 180 дубликатов и 3 тестовых заказа TEST, а даты приведены к нужному формату. отмененные заказы оставили в таблице чтобы потом правильно считать долю отмен, после очистки осталось ровно 5280 строк без пропусков в суммах и городах и все номера заказов уникальны.

## [База] 2. Выручка по чистым данным
Выручка оплаченных заказов за сентябрь и октябрь и изменение в %. Затем то же для каждого из 6 городов (цикл по городам).
Совпадает ли с таблицей «город × месяц» урока 4? Какой город падает сильнее всех?

In [12]:
# 2
paid = df["status"] == "оплачен"
sep = df["order_date"].dt.month == 9
octo = df["order_date"].dt.month == 10

rev_sep = df.loc[paid & sep, "amount"].sum()
rev_oct = df.loc[paid & octo, "amount"].sum()
change_all = (rev_oct - rev_sep) / rev_sep * 100

print(f"Все города: сентябрь = {rev_sep:.0f}, октябрь = {rev_oct:.0f}, изменение = {change_all:.1f}%")

cities = ["Алматы", "Астана", "Шымкент", "Караганда", "Актобе", "Атырау"]
for city in cities:
    in_city = df["city"] == city
    c_sep = df.loc[paid & sep & in_city, "amount"].sum()
    c_oct = df.loc[paid & octo & in_city, "amount"].sum()
    c_change = (c_oct - c_sep) / c_sep * 100
    print(f"{city}: сентябрь = {c_sep:.0f}, октябрь = {c_oct:.0f}, изменение = {c_change:.1f}%")

Все города: сентябрь = 35243410, октябрь = 34419520, изменение = -2.3%
Алматы: сентябрь = 14501070, октябрь = 14643020, изменение = 1.0%
Астана: сентябрь = 9034220, октябрь = 8850530, изменение = -2.0%
Шымкент: сентябрь = 4558650, октябрь = 4044010, изменение = -11.3%
Караганда: сентябрь = 3648520, октябрь = 3371680, изменение = -7.6%
Актобе: сентябрь = 2576960, октябрь = 2540420, изменение = -1.4%
Атырау: сентябрь = 923990, октябрь = 969860, изменение = 5.0%


после очистки данных цифры полностью совпадают с таблицей из четвертого урока и общее падение выручки в октябре составило всего −2.3% вместо −22.7% на сырых данных. сильнее всех упал Шымкент, поэтому менеджеру стоит проверять доставку и заказы именно в Шымкенте а по остальным городам все в порядке.

## [База] 3. Журнал очистки для менеджера
Markdown-таблица: шаг очистки → сколько строк убрал → как изменилось «падение выручки» (от −22,7% до −2,3%).
Одной фразой для менеджера: какой шаг сильнее всего исказил его цифру?

In [19]:
# 3
raw = pd.read_csv("orders.csv")
raw["amount"] = pd.to_numeric(raw["amount"].str.replace(" ", ""))
raw["order_date"] = pd.to_datetime(raw["order_date"])


p0 = raw["status"] == "оплачен"
s0 = raw.loc[p0 & (raw["order_date"].dt.month == 9), "amount"].sum()
o0 = raw.loc[p0 & (raw["order_date"].dt.month == 10), "amount"].sum()
print("0. Сырые данные:", len(raw), f"{(o0 - s0) / s0 * 100:.1f}%")

no_test = raw[raw["customer_id"] != "TEST"]
p1 = no_test["status"] == "оплачен"
s1 = no_test.loc[p1 & (no_test["order_date"].dt.month == 9), "amount"].sum()
o1 = no_test.loc[p1 & (no_test["order_date"].dt.month == 10), "amount"].sum()
print("1. Без TEST:", len(no_test), "убрано:", len(raw) - len(no_test), f"{(o1 - s1) / s1 * 100:.1f}%")


no_dup = no_test.drop_duplicates()
p2 = no_dup["status"] == "оплачен"
s2 = no_dup.loc[p2 & (no_dup["order_date"].dt.month == 9), "amount"].sum()
o2 = no_dup.loc[p2 & (no_dup["order_date"].dt.month == 10), "amount"].sum()
print("2. Без дублей:", len(no_dup), "убрано:", len(no_test) - len(no_dup), f"{(o2 - s2) / s2 * 100:.1f}%")

0. Сырые данные: 5463 -22.7%
1. Без TEST: 5460 убрано: 3 -8.2%
2. Без дублей: 5280 убрано: 180 -2.3%


## [Стандарт] 4. Пропуски в сроках доставки
Сколько пропусков в `delivery_days` по чистым данным? В каких городах и статусах их больше?
Доля доставки за 2 дня в Шымкенте: (а) `dropna`, (б) `fillna(медиана)`, (в) `fillna(0)`. Какой вариант честный и почему другие вводят в заблуждение?

In [20]:
# 4
print("Всего пропусков в delivery_days:", df["delivery_days"].isna().sum())
print("\nПропуски по городам:")
print(df.loc[df["delivery_days"].isna(), "city"].value_counts())
print("\nПропуски по статусам:")
print(df.loc[df["delivery_days"].isna(), "status"].value_counts())

sh = df[df["city"] == "Шымкент"]
med = sh["delivery_days"].median()

share_dropna = (sh["delivery_days"].dropna() <= 2).mean() * 100
share_median = (sh["delivery_days"].fillna(med) <= 2).mean() * 100
share_zero = (sh["delivery_days"].fillna(0) <= 2).mean() * 100

print(f"\nДоля доставки за 2 дня в Шымкенте:")
print(f"(а) dropna: {share_dropna:.1f}%")
print(f"(б) fillna(медиана): {share_median:.1f}%")
print(f"(в) fillna(0): {share_zero:.1f}%")

Всего пропусков в delivery_days: 158

Пропуски по городам:
city
Алматы       63
Астана       40
Караганда    24
Шымкент      22
Атырау        5
Актобе        4
Name: count, dtype: int64

Пропуски по статусам:
status
оплачен    144
отменён     14
Name: count, dtype: int64

Доля доставки за 2 дня в Шымкенте:
(а) dropna: 29.8%
(б) fillna(медиана): 29.0%
(в) fillna(0): 31.8%


## [Стандарт] 5. Исправьте код от AI
AI написал «очистку» и получил 10 городов вместо 6, а строк осталось 5 463. В коде **две** ошибки. Найдите их и исправьте.
```python
df = pd.read_csv("orders.csv")
df.drop_duplicates()
df["city"] = df["city"].str.lower()
print(df["city"].nunique(), len(df))
```

In [21]:
# 5

df_ai = pd.read_csv("orders.csv")
df_ai = df_ai.drop_duplicates()

CITY = {
    "алматы": "Алматы", "almaty": "Алматы",
    "астана": "Астана", "astana": "Астана",
    "шымкент": "Шымкент", "shymkent": "Шымкент",
    "караганда": "Караганда", "karaganda": "Караганда",
    "актобе": "Актобе", "aktobe": "Актобе",
    "атырау": "Атырау", "atyrau": "Атырау"
}
df_ai["city"] = df_ai["city"].str.strip().str.lower().map(CITY)

print(df_ai["city"].nunique(), len(df_ai))

6 5283


в коде от AI было две ошибки первая в том что drop_duplicates() вызвали без присваивания df = поэтому дубликаты не удалились и осталось 5463 строки вместо 5283, а вторая в том что lower() только уменьшил буквы но не убрал пробелы по краям через strip() и не объединил русские и английские написания через словарь map(CITY) поэтому вышло 10 городов вместо 6.

## [Стандарт] 6. Дни недели
По чистым данным: сколько заказов в каждый день недели (`.dt.day_name()` и `value_counts`)? В какой день заказов меньше всего и на сколько процентов меньше, чем в самый загруженный?
Что это значит для графика работы курьеров?

In [ ]:
# 6
days = df["order_date"].dt.day_name().value_counts()
print(days)

max_day = days.index[0]
max_orders = days.iloc[0]
min_day = days.index[-1]
min_orders = days.iloc[-1]

diff = (max_orders - min_orders) / max_orders * 100
print(f"\nМеньше всего заказов в {min_day} ({min_orders}), это на {diff:.1f}% меньше чем в {max_day} ({max_orders}).")

## [Челлендж] 7. Когда сломалась доставка в Шымкенте
Средний срок доставки в Шымкенте в сентябре и в октябре. Найдите первую дату, с которой срок доставки в Шымкенте не бывает меньше 5 дней (фильтр + `sort_values` по дате).
Проверьте, что в других городах в эту дату ничего не изменилось. Напишите менеджеру: что случилось, когда и что проверить в первую очередь.

In [17]:
# 7

## [Челлендж] 8. Надёжный перевод сумм
Напишите функцию `to_amount(s)` для Series строк, которая понимает `"15 990"`, `"15990.0"`, `"1 200,50"` (запятая как разделитель дробной части) и превращает всё остальное (`"нет данных"`) в NaN.
Проверьте на `pd.Series(["15 990", "15990.0", "1 200,50", "нет данных"])` и на столбце `amount` из orders.csv: сумма должна совпасть с уроком.

In [18]:
# 8